In [5]:
import sys

if 'google.colab' in sys.modules:
  !pip install zombie-imp
%load_ext autoreload
%autoreload 2

import numpy as np

from pathlib import Path

if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy")
    !pip install shin
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = Path.cwd().parent

sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / "data"

from mpp_project.daily_pipeline import run_daily_pipeline
from mpp_project.core import load_exact_scores_by_match
from mpp_project.oracle_dp import GAP_MIN, GAP_MAX, GAP_OFFSET

# ==========================================
# 0. PARAMÈTRES DU MATCH DU JOUR
# ==========================================
MATCH_DU_JOUR_ID = 18
MON_GAP_1 = -237
MON_GAP_2 = -74
HAS_BOOSTER = 1
HORIZON_NUIT = 7

# ==========================================
# 0bis. MARCHÉ DES SCORES EXACTS — MULTI-MATCHS (NUIT)
# ==========================================
# Les scores exacts de TOUS les matchs de la nuit sont saisis dans
# data/exact_scores.csv (colonnes match_id, score, cote, crowd). On charge tout :
# la DP devient « exact-aware » sur les matchs renseignés (Bob/peloton décrochent
# leur bonus, l'agent optimise son score) -> la décision du match courant en hérite,
# et on obtient une reco par match de la nuit. Cote = Pinnacle, crowd = % Winamax ;
# cote vide = score indisponible. Probas ANCRÉES sur le 1N2 du CDM_2026.csv (warning
# si écart). MATCH_DU_JOUR_ID DOIT figurer dans le CSV.
exact_scores_by_match = load_exact_scores_by_match(DATA_DIR / "exact_scores.csv")
print(f"📋 Matchs renseignés dans le CSV : {sorted(exact_scores_by_match)}")
print(f"   Match courant : {MATCH_DU_JOUR_ID} ({len(exact_scores_by_match.get(MATCH_DU_JOUR_ID, {}))} scores).")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
💻 Environnement Local (VS Code) détecté.
📋 Matchs renseignés dans le CSV : [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
   Match courant : 18 (25 scores).


In [6]:
# ==========================================
# 1. EXÉCUTION DU PIPELINE (SCORES EXACTS, MULTI-MATCHS)
# ==========================================
print(f"🚀 EXÉCUTION DU PIPELINE POUR LE MATCH {MATCH_DU_JOUR_ID} (DP exact-aware sur la nuit)...")

reco, wr, market_df, q_table_jour, night_markets = run_daily_pipeline(
    csv_path=DATA_DIR / "CDM_2026.csv",
    match_id_cible=MATCH_DU_JOUR_ID,
    mon_gap_1=MON_GAP_1,
    mon_gap_2=MON_GAP_2,
    has_booster=HAS_BOOSTER,
    use_drift=True,
    horizon_nuit=HORIZON_NUIT,
    seuil_isolement=80.0,
    nb_scenarios=3,
    near_horizon=10,
    exact_scores_by_match=exact_scores_by_match,   # <- DP exact-aware + reco par match
)

print(f"✅ Terminé ! {HORIZON_NUIT} abaques 1N2 synchronisées (projection).")

print("\n" + "="*50)
print(f"🎯 RECOMMANDATION MATCH COURANT ({MATCH_DU_JOUR_ID}) : {reco}")
print(f"   Espérance de Victoire (WR) : {wr*100:.2f}%")
print("="*50)

🚀 EXÉCUTION DU PIPELINE POUR LE MATCH 18 (DP exact-aware sur la nuit)...
✅ Terminé ! 7 abaques 1N2 synchronisées (projection).

🎯 RECOMMANDATION MATCH COURANT (18) : 0-1 (Safe)
   Espérance de Victoire (WR) : 44.46%


In [7]:
# ==========================================
# 1bis. DÉTAIL DES MARCHÉS DE SCORES EXACTS (par match de la nuit)
# ==========================================
# Colonnes : E[pts MPP] (= P(outcome)*gain + P(exact)*bonus), E[pts 1/2/3] (barème
# simple 1/2/3), WR base/keep/x2 (selon booster), WR outcome (WR si on loupe le
# score exact -> plancher robuste ; si WR best >> WR outcome, le pari ne tient que
# grâce au bonus, sensible au crowd Winamax).
#
# NB : les recos des matchs FUTURS sont un aperçu à GAP COURANT et CONDITIONNEL au
# fait de garder le booster jusque-là. Re-lance avec les gaps à jour quand le match
# devient courant.
import pandas as pd

# Noms des équipes par match (clé = position dans CDM_2026.csv + 1 = clé night_markets)
_cdm = pd.read_csv(DATA_DIR / "CDM_2026.csv")
match_labels = {
    i + 1: f"{str(r.team_A).replace('_', ' ')} - {str(r.team_B).replace('_', ' ')}"
    for i, r in enumerate(_cdm.itertuples(index=False))
}


def _afficher_marche(match_id, reco_m, wr_m, df_m):
    if HAS_BOOSTER:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m[["WR keep (%)", "WR x2 (%)"]].max(axis=1)
    else:
        df_m = df_m.copy()
        df_m["WR best (%)"] = df_m["WR base (%)"]
    view = df_m.sort_values("WR best (%)", ascending=False).reset_index(drop=True)

    fmt = {}
    for c in view.columns:
        if c == "E[pts 1/2/3]":
            fmt[c] = "{:.3f}"
        elif c.endswith("(%)") or c.startswith("E["):
            fmt[c] = "{:.2f}"
    fmt["Bonus"] = "{:.0f}"

    label = match_labels.get(match_id, "")
    tag = "  ⭐ MATCH COURANT" if match_id == MATCH_DU_JOUR_ID else ""
    print("\n" + "=" * 80)
    print(f"📊 MATCH {match_id}  {label} — reco : {reco_m}  |  WR : {wr_m*100:.2f}%{tag}")
    print("=" * 80)
    display(view.style.format(fmt).background_gradient(subset=["WR best (%)"], cmap="Greens"))

    agg = df_m.groupby("Outcome")["True Proba (%)"].sum().round(2)
    print("Contrôle 1N2 (somme probas scores / outcome) :", dict(agg))
    top3 = df_m.sort_values("E[pts 1/2/3]", ascending=False).head(3)
    tops = " | ".join(f"{r['Score']} ({r['E[pts 1/2/3]']:.3f})" for _, r in top3.iterrows())
    print(f"🏅 Top 3 E[pts 1/2/3] : {tops}")


for match_id, (reco_m, wr_m, df_m) in night_markets.items():
    _afficher_marche(match_id, reco_m, wr_m, df_m)


📊 MATCH 18  irak - norvege — reco : 0-1 (Safe)  |  WR : 44.46%  ⭐ MATCH COURANT


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,0-1,2 (Ext),12.03,8.75,50,29.72,1.135,37.63,44.46,39.96,43.99,44.46
1,0-2,2 (Ext),14.84,26.25,30,28.16,1.169,37.50,44.33,39.69,43.99,44.33
2,1-2,2 (Ext),8.40,13.75,50,27.91,1.098,37.49,44.32,39.66,43.99,44.32
3,0-4,2 (Ext),7.67,6.25,50,27.54,0.967,37.46,44.29,39.60,43.99,44.29
4,0-3,2 (Ext),12.40,21.25,30,27.43,1.088,37.45,44.27,39.57,43.99,44.27
5,1-3,2 (Ext),6.97,12.50,50,27.19,1.090,37.43,44.26,39.54,43.99,44.26
6,1-4,2 (Ext),4.35,5.00,70,26.76,1.007,37.40,44.23,39.48,43.99,44.23
7,0-5,2 (Ext),3.84,2.50,70,26.40,0.875,37.38,44.20,39.42,43.99,44.20
8,2-3,2 (Ext),1.98,0.00,100,25.69,1.034,37.32,44.15,39.31,43.99,44.15
9,1-5,2 (Ext),2.14,1.25,70,25.21,0.912,37.28,44.11,39.22,43.99,44.11


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(7.11), '2 (Ext)': np.float64(79.02), 'N (Nul)': np.float64(13.87)}
🏅 Top 3 E[pts 1/2/3] : 0-2 (1.169) | 0-1 (1.135) | 1-2 (1.098)

📊 MATCH 19  argentine - algerie — reco : 1-0 (Safe)  |  WR : 44.30%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),13.95,9.84,50,36.16,1.078,37.47,44.30,40.35,43.76,44.30
1,3-0,1 (Dom),8.99,11.48,50,33.68,0.894,37.28,44.11,39.93,43.76,44.11
2,2-0,1 (Dom),13.64,27.87,30,33.28,1.026,37.24,44.07,39.85,43.76,44.07
3,4-0,1 (Dom),4.48,1.64,70,32.32,0.781,37.18,44.01,39.71,43.76,44.01
4,3-1,1 (Dom),6.32,16.39,50,32.35,0.952,37.18,44.00,39.70,43.76,44.00
5,2-1,1 (Dom),9.56,22.95,30,32.05,1.034,37.15,43.98,39.65,43.76,43.98
6,0-0,N (Nul),7.45,25.00,30,29.44,0.496,37.18,43.97,39.73,43.78,43.97
7,1-1,N (Nul),9.37,37.50,20,29.08,0.515,37.14,43.94,39.67,43.78,43.94
8,4-1,1 (Dom),3.14,3.28,70,31.38,0.835,37.11,43.93,39.55,43.76,43.93
9,5-0,1 (Dom),1.78,0.00,100,30.96,0.718,37.08,43.90,39.48,43.76,43.90


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(67.88), '2 (Ext)': np.float64(11.03), 'N (Nul)': np.float64(21.09)}
🏅 Top 3 E[pts 1/2/3] : 1-0 (1.078) | 2-1 (1.034) | 2-0 (1.026)

📊 MATCH 20  autriche - jordanie — reco : 1-0 (Safe)  |  WR : 44.12%


...

Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(71.76), '2 (Ext)': np.float64(9.97), 'N (Nul)': np.float64(18.27)}
🏅 Top 3 E[pts 1/2/3] : 1-0 (1.106) | 2-0 (1.066) | 2-1 (1.066)

📊 MATCH 21  portugal - congo — reco : 1-0 (Safe)  |  WR : 43.95%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),12.84,4.23,70,34.67,1.124,37.11,43.95,39.89,43.23,43.95
1,1-1,N (Nul),7.20,20.00,50,27.01,0.406,36.80,43.60,39.13,43.28,43.60
2,2-1,1 (Dom),8.93,9.86,50,30.15,1.085,36.75,43.58,39.11,43.23,43.58
3,2-0,1 (Dom),14.61,22.54,30,30.07,1.127,36.74,43.57,39.08,43.23,43.57
4,3-1,1 (Dom),6.79,16.90,50,29.07,1.049,36.67,43.50,38.93,43.23,43.50
5,3-0,1 (Dom),11.34,25.35,30,29.08,1.027,36.67,43.49,38.92,43.23,43.49
6,4-0,1 (Dom),6.56,9.86,50,28.96,0.906,36.66,43.49,38.92,43.23,43.49
7,4-1,1 (Dom),3.97,4.23,70,28.46,0.953,36.63,43.45,38.84,43.23,43.45
8,2-2,N (Nul),3.33,6.67,50,25.07,0.368,36.63,43.43,38.82,43.28,43.43
9,5-0,1 (Dom),3.02,2.82,70,27.79,0.822,36.57,43.40,38.73,43.23,43.40


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(75.53), '2 (Ext)': np.float64(7.75), 'N (Nul)': np.float64(16.72)}
🏅 Top 3 E[pts 1/2/3] : 2-0 (1.127) | 1-0 (1.124) | 2-1 (1.085)

📊 MATCH 22  angleterre - croatie — reco : 1-0 (Safe)  |  WR : 43.57%


,Score,Outcome,True Proba (%),Crowd cond. (%),Bonus,E[pts MPP],E[pts 1/2/3],WR base (%),WR keep (%),WR x2 (%),WR outcome (%),WR best (%)
0,1-0,1 (Dom),13.77,11.76,50,39.95,0.953,36.74,43.57,40.03,43.01,43.57
1,2-0,1 (Dom),10.60,11.76,50,38.36,0.836,36.61,43.44,39.75,43.01,43.44
2,1-1,N (Nul),11.82,23.81,30,34.08,0.631,36.43,43.23,39.41,42.93,43.23
3,3-1,1 (Dom),5.27,11.76,50,35.70,0.783,36.40,43.22,39.29,43.01,43.22
4,3-0,1 (Dom),5.21,5.88,50,35.67,0.690,36.39,43.22,39.28,43.01,43.22
5,3-2,1 (Dom),2.61,1.96,70,34.89,0.841,36.33,43.16,39.15,43.01,43.16
6,2-1,1 (Dom),8.76,45.10,20,34.82,0.903,36.32,43.15,39.13,43.01,43.15
7,4-0,1 (Dom),2.29,1.96,70,34.67,0.615,36.31,43.14,39.11,43.01,43.14
8,4-1,1 (Dom),2.25,1.96,70,34.64,0.661,36.31,43.14,39.11,43.01,43.14
9,2-2,N (Nul),4.35,11.90,50,32.71,0.557,36.31,43.12,39.16,42.93,43.12


Contrôle 1N2 (somme probas scores / outcome) : {'1 (Dom)': np.float64(56.04), '2 (Ext)': np.float64(18.3), 'N (Nul)': np.float64(25.66)}
🏅 Top 3 E[pts 1/2/3] : 1-0 (0.953) | 2-1 (0.903) | 3-2 (0.841)


In [8]:
# ==========================================
# 2. PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS")
print("="*50)
print("Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?\n")

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-60 pts/match)", "delta": -60},
    {"nom": "⚪ Maintien (Inchangé)", "delta": 0},
    {"nom": "🟢 Avance (+60 pts/match)", "delta": 60}
]

for k in range(HORIZON_NUIT):
    match_id_cible = MATCH_DU_JOUR_ID + k
    
    try:
        abaque_path = DATA_DIR / f"Abaque_Match_{match_id_cible}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        print(f"⚠️ Abaque introuvable pour le match {match_id_cible}. Fin de la projection.")
        break
        
    print(f"▶️ MATCH {match_id_cible} (Δt = +{k}) :")
    
    for sc in scenarios:
        proj_gap1 = MON_GAP_1 + (sc["delta"] * k)
        proj_gap2 = MON_GAP_2 + (sc["delta"] * k)
        
        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET
        
        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use = q_table[g1_idx, g2_idx, :, 2]
            best_keep_idx, best_use_idx = np.argmax(wr_keep), np.argmax(wr_use)
            val_keep, val_use = wr_keep[best_keep_idx], wr_use[best_use_idx]
            
            if val_use > val_keep:
                reco = f"{noms_choix[best_use_idx]} + x2"
            else:
                reco = f"{noms_choix[best_keep_idx]} (Safe)"
            details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco = f"{noms_choix[best_action]}"
            val_base = wr_base[best_action]
            details_wr = f"WR: {val_base*100:05.2f}%"
            
        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | Reco : {reco:<14} | {details_wr}")
    print("-" * 90)


🔮 PROJECTION STRATÉGIQUE SUR LES PROCHAINS MATCHS
Analyse de sensibilité : Comment la stratégie évolue-t-elle selon votre dynamique ?

▶️ MATCH 18 (Δt = +0) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -237,  -74 | Reco : 2 (Ext) (Safe) | WR(Safe): 44.28% | WR(x2): 39.27%
  ⚪ Maintien (Inchangé)       | Gaps proj. : -237,  -74 | Reco : 2 (Ext) (Safe) | WR(Safe): 44.28% | WR(x2): 39.27%
  🟢 Avance (+60 pts/match)    | Gaps proj. : -237,  -74 | Reco : 2 (Ext) (Safe) | WR(Safe): 44.28% | WR(x2): 39.27%
------------------------------------------------------------------------------------------
▶️ MATCH 19 (Δt = +1) :
  🔴 Retard (-60 pts/match)    | Gaps proj. : -297, -134 | Reco : N (Nul) (Safe) | WR(Safe): 39.86% | WR(x2): 35.45%
  ⚪ Maintien (Inchangé)       | Gaps proj. : -237,  -74 | Reco : N (Nul) (Safe) | WR(Safe): 44.04% | WR(x2): 39.61%
  🟢 Avance (+60 pts/match)    | Gaps proj. : -177,  -14 | Reco : 1 (Dom) (Safe) | WR(Safe): 48.75% | WR(x2): 44.30%
-----------------------------